# 02 — Call structure

## Manuscript crosswalk

- **Methods:** Call structure analysis; spectro-temporal parameters (STP), MFCC, exact constrained DTW, and VAE; repertoire aggregation; descriptive embeddings.
- **Results:** eligible call repertoires and comparisons; absence of a resolved stage-related call-structure change.
- **Figures:** Figures S5–S8 (MFCC, STP, DTW, and VAE call spaces).

This notebook validates and describes the four call representations used in the manuscript. It reads their versioned caches; it never recalculates the all-call DTW matrix or retrains the VAE.

## Cached artifacts

Required quantitative inputs:

- `data/processed/calls.csv`
- `data/cache/call_session_inventory.csv`
- `data/cache/stp_pca_scores.csv`
- `data/cache/mfcc_pca_scores.csv`
- `data/cache/dtw_distances_condensed.npz`
- `data/cache/vae_latent_means.csv`
- `data/derived/call_repertoire_distances.csv`
- `data/cache/embeddings/{mfcc,stp,dtw,vae}.csv`

The methods and fixed parameters are recorded in [the analysis specification](../docs/analysis_specification.md). The optional unsuffixed PNGs are provenance-locked manuscript snapshots, not computed outputs of this notebook. They are displayed only when present and registered; their presence cannot replace a missing quantitative cache. The same cheap plotting path is available without Jupyter through [`scripts/regenerate_cached_figures.py`](../scripts/regenerate_cached_figures.py).

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside a clone containing README.md and data/."
    )


ROOT = find_repo_root()
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from marmoset_convergence.cli import validate_cache
from marmoset_convergence.figures import (
    EMBEDDING_FIGURES,
    prepare_embedding_table,
    regenerate_embedding_space,
)
from marmoset_convergence.provenance import CacheManifest

EXPENSIVE_STEPS = {
    "recompute_sequence": False,
    "recompute_dtw": False,
    "retrain_vae": False,
    "refit_models": False,
}
if any(EXPENSIVE_STEPS.values()):
    raise RuntimeError(
        "This notebook is cached-only. Expensive work requires "
        "'bash scripts/run_repro_pipeline.sh --mode full'."
    )

pd.DataFrame({"step": EXPENSIVE_STEPS.keys(), "enabled": EXPENSIVE_STEPS.values()})

In [ ]:
MANIFEST_PATHS = (
    ROOT / "data/processed/manifest.json",
    ROOT / "data/cache/manifest.json",
    ROOT / "data/derived/manifest.json",
    ROOT / "results/manifest.json",
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_manifests() -> list[tuple[Path, dict]]:
    found = []
    for path in MANIFEST_PATHS:
        if path.is_file():
            with path.open(encoding="utf-8") as handle:
                found.append((path, json.load(handle)))
    if not found:
        raise FileNotFoundError("No provenance manifest was found.")
    return found


def iter_records(payload: dict):
    for section_name in ("artifacts", "sources", "inputs", "files"):
        section = payload.get(section_name, {})
        if isinstance(section, dict):
            for key, value in section.items():
                record = value if isinstance(value, dict) else {"sha256": value}
                yield str(record.get("path", key)), record
        elif isinstance(section, list):
            for record in section:
                if isinstance(record, dict) and record.get("path"):
                    yield str(record["path"]), record


def record_sha256(record: dict) -> str | None:
    value = record.get("sha256") or record.get("checksum_sha256")
    if value:
        return str(value).removeprefix("sha256:")
    checksum = record.get("checksum")
    if isinstance(checksum, str):
        return checksum.removeprefix("sha256:")
    if isinstance(checksum, dict) and checksum.get("algorithm", "").lower() == "sha256":
        return checksum.get("value")
    return None


MANIFESTS = load_manifests()


def registered_artifact(relative_path: str) -> tuple[Path, dict]:
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Required cached artifact is missing: {relative_path}. No expensive fallback will run."
        )
    normalized = Path(relative_path).as_posix()
    for manifest_path, payload in MANIFESTS:
        for recorded_path, record in iter_records(payload):
            recorded = Path(recorded_path)
            same_path = (recorded.resolve() == path.resolve()) if recorded.is_absolute() else (recorded.as_posix().lstrip("./") == normalized)
            if same_path:
                expected = record_sha256(record)
                if not expected:
                    raise RuntimeError(f"No SHA-256 for {relative_path} in {manifest_path}.")
                observed = sha256_file(path)
                if observed.lower() != expected.lower():
                    raise RuntimeError(f"Checksum mismatch for {relative_path}.")
                return path, record
    raise RuntimeError(f"{relative_path} is not registered in a provenance manifest.")


def require_parameter(record: dict, name: str, expected: object) -> None:
    parameters = record.get("parameters")
    if not isinstance(parameters, dict) or name not in parameters:
        raise RuntimeError(f"Manifest does not record required parameter {name!r}.")
    observed = parameters[name]
    if isinstance(expected, float):
        matches = np.isclose(float(observed), expected)
    else:
        matches = observed == expected
    if not matches:
        raise RuntimeError(f"Manifest parameter {name}={observed!r}; expected {expected!r}.")

## Validate unique-call representations

STP uses five principal components from 11 standardized parameters. MFCC uses five principal components from 132 standardized coefficients. To preserve the basis underlying the reported results, both caches reuse scores from PCA fitted to the 3,907-row historical merged table before three unavailable-audio rows were omitted and before exact-filename deduplication; they expose one row for each of the 3,612 canonical calls. DTW uses exact dynamic programming with a normalized-time band of 0.20 and path-length normalization. The VAE supplies a 32-dimensional latent mean for each unique call.

In [ ]:
calls_path, _ = registered_artifact("data/processed/calls.csv")
call_inventory_path, _ = registered_artifact("data/cache/call_session_inventory.csv")
stp_path, _ = registered_artifact("data/cache/stp_pca_scores.csv")
mfcc_path, _ = registered_artifact("data/cache/mfcc_pca_scores.csv")
vae_path, vae_record = registered_artifact("data/cache/vae_latent_means.csv")
dtw_path, dtw_record = registered_artifact("data/cache/dtw_distances_condensed.npz")

calls = pd.read_csv(calls_path)
call_inventory = pd.read_csv(call_inventory_path)
stp_scores = pd.read_csv(stp_path)
mfcc_scores = pd.read_csv(mfcc_path)
vae_latents = pd.read_csv(vae_path)
required_inventory = {"repertoire_id", "n_calls", "pair_id", "stage", "context"}
if missing := sorted(required_inventory.difference(call_inventory.columns)):
    raise ValueError(f"Call inventory is missing columns: {missing}")

for label, table in {
    "calls": calls,
    "STP scores": stp_scores,
    "MFCC scores": mfcc_scores,
    "VAE latents": vae_latents,
}.items():
    if "call_id" not in table:
        raise ValueError(f"{label} lacks call_id.")
    if table["call_id"].duplicated().any():
        raise ValueError(f"{label} contains duplicate call_id values.")


def representation_columns(table: pd.DataFrame, expected_n: int, pattern: str) -> list[str]:
    named = [column for column in table if re.fullmatch(pattern, column, flags=re.IGNORECASE)]
    if len(named) == expected_n:
        return named
    numeric = [
        column for column in table.select_dtypes(include="number").columns
        if column not in {"session_number", "phase"}
    ]
    if len(numeric) == expected_n:
        return numeric
    raise ValueError(
        f"Expected {expected_n} representation columns, found named={len(named)} "
        f"and numeric={len(numeric)}."
    )


stp_columns = representation_columns(stp_scores, 5, r"(?:stp_?)?pc_?0*[1-5]")
mfcc_columns = representation_columns(mfcc_scores, 5, r"(?:mfcc_?)?pc_?0*[1-5]")
latent_columns = representation_columns(
    vae_latents, 32, r"(?:(?:vae_)?latent|mu|z)_?0*(?:[1-9]|[12][0-9]|3[0-2])"
)

expected_call_ids = calls["call_id"].astype(str).to_numpy()
for label, table in {
    "STP": stp_scores,
    "MFCC": mfcc_scores,
    "VAE": vae_latents,
}.items():
    if set(table["call_id"].astype(str)) != set(expected_call_ids):
        raise ValueError(f"{label} call_id set does not match data/processed/calls.csv.")

with np.load(dtw_path, allow_pickle=False) as archive:
    required_keys = {"call_ids", "distances"}
    missing_keys = sorted(required_keys.difference(archive.files))
    if missing_keys:
        raise ValueError(f"DTW cache lacks canonical NPZ keys: {missing_keys}")
    dtw_call_ids = archive["call_ids"].astype(str)
    dtw_distances = archive["distances"]
    if not np.array_equal(dtw_call_ids, expected_call_ids):
        raise ValueError("DTW call_ids do not exactly match the canonical call-table order.")
    if dtw_distances.shape != (6_521_466,):
        raise ValueError(f"Unexpected DTW condensed shape: {dtw_distances.shape}")
    if not np.isfinite(dtw_distances).all() or (dtw_distances < 0).any():
        raise ValueError("DTW distances must be finite and non-negative.")

for name, expected in {
    "algorithm": "exact_dynamic_programming",
    "frequency_min_hz": 4000,
    "frequency_max_hz": 12000,
    "window_samples": 2048,
    "hop_samples": 512,
    "db_min": -80,
    "db_max": 0,
    "local_cost": "cosine",
    "normalized_time_constraint": 0.20,
    "path_normalization": "alignment_steps",
    "storage": "condensed_scipy",
}.items():
    require_parameter(dtw_record, name, expected)

for name, expected in {
    "implementation": "chatter",
    "version": "0.1.6",
    "image_size": [128, 128],
    "canvas_seconds": 2.5,
    "latent_dim": 32,
    "epochs": 100,
    "batch_size": 32,
    "learning_rate": 0.0001,
    "beta": 0.5,
    "seed": 42,
}.items():
    require_parameter(vae_record, name, expected)

representation_audit = pd.DataFrame([
    {"representation": "STP PCA", "expected_calls": 3612, "observed_calls": len(stp_scores),
     "expected_dimensions": 5, "observed_dimensions": len(stp_columns)},
    {"representation": "MFCC PCA", "expected_calls": 3612, "observed_calls": len(mfcc_scores),
     "expected_dimensions": 5, "observed_dimensions": len(mfcc_columns)},
    {"representation": "Exact DTW", "expected_calls": 3612, "observed_calls": len(dtw_call_ids),
     "expected_dimensions": 6_521_466, "observed_dimensions": len(dtw_distances)},
    {"representation": "VAE latent mean", "expected_calls": 3612, "observed_calls": len(vae_latents),
     "expected_dimensions": 32, "observed_dimensions": len(latent_columns)},
])
representation_audit["passes"] = (
    representation_audit["expected_calls"].eq(representation_audit["observed_calls"])
    & representation_audit["expected_dimensions"].eq(representation_audit["observed_dimensions"])
)
display(representation_audit)
if not representation_audit["passes"].all():
    raise AssertionError("Call-representation audit failed.")

## Validate repertoire distances and embeddings

For each metric, the distance between two eligible session repertoires is the arithmetic mean of every cross-repertoire call-to-call distance. The two-dimensional embeddings are descriptive and never replace these original-space distances in the model.

In [ ]:
distance_path, _ = registered_artifact("data/derived/call_repertoire_distances.csv")
call_pair_path, _ = registered_artifact("data/cache/call_session_pairs.csv")
call_distances = pd.read_csv(distance_path)
call_pair_index = pd.read_csv(call_pair_path)
required = {"comparison_id", "context", "metric", "distance", "repertoire_a_id", "repertoire_b_id", "n_lower_level_pairs"}
if missing := sorted(required.difference(call_distances.columns)):
    raise ValueError(f"Call-distance table is missing columns: {missing}")
if (call_distances["distance"] < 0).any() or not np.isfinite(call_distances["distance"]).all():
    raise ValueError("Call repertoire distances must be finite and non-negative.")
lower_level_pairs = pd.to_numeric(call_distances["n_lower_level_pairs"], errors="coerce")
if (
    lower_level_pairs.isna().any()
    or (lower_level_pairs <= 0).any()
    or not np.equal(lower_level_pairs, np.floor(lower_level_pairs)).all()
):
    raise ValueError("n_lower_level_pairs must contain positive integers.")
call_distances["n_lower_level_pairs"] = lower_level_pairs.astype(np.int64)

pair_key = ["comparison_id", "repertoire_a_id", "repertoire_b_id"]
if missing := sorted(set(pair_key).difference(call_pair_index.columns)):
    raise ValueError(f"Call pair index is missing columns: {missing}")
if call_pair_index["comparison_id"].duplicated().any():
    raise ValueError("Call pair index contains duplicate comparison_id values.")
distance_pairs = call_distances[pair_key].drop_duplicates()
if distance_pairs["comparison_id"].duplicated().any():
    raise ValueError("A call comparison maps to inconsistent repertoire IDs.")
pair_membership = call_pair_index[pair_key].merge(
    distance_pairs, on=pair_key, how="outer", indicator=True, validate="one_to_one"
)
if not pair_membership["_merge"].eq("both").all():
    raise AssertionError("Call repertoire distances do not exactly cover the registered pair index.")

inventory_counts = pd.to_numeric(call_inventory["n_calls"], errors="coerce")
if (
    inventory_counts.isna().any()
    or (inventory_counts <= 0).any()
    or not np.equal(inventory_counts, np.floor(inventory_counts)).all()
):
    raise ValueError("Call inventory n_calls values must be positive integers.")
inventory_counts = pd.Series(
    inventory_counts.to_numpy(dtype=np.int64), index=call_inventory["repertoire_id"]
)
expected_products = call_pair_index[pair_key].copy()
expected_products["expected_n_lower_level_pairs"] = (
    expected_products["repertoire_a_id"].map(inventory_counts)
    * expected_products["repertoire_b_id"].map(inventory_counts)
)
if expected_products["expected_n_lower_level_pairs"].isna().any():
    raise ValueError("Call pair index references a repertoire absent from the inventory.")
observed_products = call_distances[["comparison_id", "n_lower_level_pairs"]].drop_duplicates()
if observed_products["comparison_id"].duplicated().any():
    raise ValueError("Call metrics disagree on n_lower_level_pairs.")
product_audit = expected_products[["comparison_id", "expected_n_lower_level_pairs"]].merge(
    observed_products, on="comparison_id", how="outer", validate="one_to_one"
)
if product_audit.isna().any().any() or not np.array_equal(
    product_audit["expected_n_lower_level_pairs"].to_numpy(dtype=np.int64),
    product_audit["n_lower_level_pairs"].to_numpy(dtype=np.int64),
):
    raise AssertionError("n_lower_level_pairs does not equal n_calls(a) × n_calls(b).")

metric_aliases = {
    "stp": "stp", "spectro-temporal": "stp", "spectro_temporal": "stp",
    "mfcc": "mfcc", "dtw": "dtw", "vae": "vae",
}
call_distances["metric_canonical"] = (
    call_distances["metric"].astype(str).str.strip().str.lower().map(metric_aliases)
)
if call_distances["metric_canonical"].isna().any():
    unknown = sorted(call_distances.loc[call_distances["metric_canonical"].isna(), "metric"].unique())
    raise ValueError(f"Unknown call metrics: {unknown}")

unique_comparisons = call_distances.drop_duplicates("comparison_id")
context = unique_comparisons["context"].astype(str).str.lower().str.replace("_", "-")
context = context.replace({"stranger": "non-partner", "nonpartner": "non-partner"})

cache_manifest = CacheManifest.load(ROOT / "data/cache/manifest.json")
embedding_rows = []
call_embeddings = {}
for metric in ("mfcc", "stp", "dtw", "vae"):
    relative_path = f"data/cache/embeddings/{metric}.csv"
    path, record = registered_artifact(relative_path)
    manifest_names = [name for name, artifact in cache_manifest.artifacts.items() if artifact.path == relative_path]
    if len(manifest_names) != 1:
        raise ValueError(f"Expected one cache-manifest record for {relative_path}; found {len(manifest_names)}.")
    validate_cache(ROOT, manifest_names)
    table = pd.read_csv(path)
    prepare_embedding_table(table, calls, id_column="call_id")
    call_embeddings[metric] = table
    embedding_rows.append({
        "embedding": metric, "rows": len(table), "columns": table.shape[1],
        "method": record["parameters"]["algorithm"],
        "package": record["parameters"]["package"],
        "seed": record["parameters"]["random_seed"],
        "fit_scope": record["parameters"]["fit_scope"],
    })

call_count_audit = pd.DataFrame([
    {"quantity": "Eligible call repertoires", "expected": 130,
     "observed": call_inventory["repertoire_id"].nunique()},
    {"quantity": "Calls in eligible repertoires", "expected": 3527,
     "observed": int(call_inventory["n_calls"].sum())},
    {"quantity": "Unique call comparisons", "expected": 431,
     "observed": unique_comparisons["comparison_id"].nunique()},
    {"quantity": "Partner call comparisons", "expected": 74,
     "observed": int((context == "partner").sum())},
    {"quantity": "Non-partner call comparisons", "expected": 357,
     "observed": int((context == "non-partner").sum())},
    {"quantity": "Call metric rows", "expected": 1724,
     "observed": len(call_distances)},
])
call_count_audit["passes"] = call_count_audit["expected"].eq(call_count_audit["observed"])
display(call_count_audit)
display(pd.DataFrame(embedding_rows))
if not call_count_audit["passes"].all():
    raise AssertionError("Call-distance manuscript count audit failed.")
if set(call_distances["metric_canonical"]) != {"stp", "mfcc", "dtw", "vae"}:
    raise AssertionError("The call-distance table does not contain all four canonical metrics.")

## Figures S5–S8

The locked manuscript panels reuse the same cached coordinates before they are divided by stage and context. Large circles, bonded-pair links, and 50% Gaussian covariance ellipses are descriptive. The regenerated exports below use those same pooled coordinates and fixed limits; they do not reconstruct the locked spectrogram-card annotations because the complete WAV collection is outside the cached-mode contract. The formal conclusions come from the original-space distances and the model in Notebook 05.

In [ ]:
figure_specs = {
    "Figure S5 — MFCC": "s5",
    "Figure S6 — STP": "s6",
    "Figure S7 — DTW": "s7",
    "Figure S8 — VAE": "s8",
}
for label, code in figure_specs.items():
    spec = EMBEDDING_FIGURES[code]
    relative_path = spec.locked_relative_path
    if (ROOT / relative_path).is_file():
        verified_path, _ = registered_artifact(relative_path)
        display(f"{label} — locked manuscript reference", Image(filename=str(verified_path)))
    else:
        print(f"{label}: optional rendered export is not present at {relative_path}.")
    regenerated_path = regenerate_embedding_space(
        call_embeddings[spec.metric], calls, spec, ROOT / relative_path,
        overwrite_regenerated=True,
    )
    display(f"{label} — regenerated analytical export", Image(filename=str(regenerated_path)))

## Result connection

All four cached acoustic spaces retain structured caller variation, but the manuscript reports no consistent contraction of bonded-pair separation after pair formation in either context. That inferential statement is evaluated from the saved unified-model posterior draws in Notebook 05, not from visual distances in Figures S5–S8.